In [ ]:
# GP (2026-08-12)
# 
# NOTEs:
#  - Might need revision: section "2.3. Valeurs brutes des indicateurs". Blocked by: https://bluesquare.atlassian.net/browse/SNT25-638 
#  - sizes of many figures could be improved (some plots could be made smaller to fit better in rendered html ... )
#  - code itself has a lot of redundnacies, could be streamlined (refactorized) ...
#  - Shapes simplification (`shapes_data_stsimplified <- st_simplify(shapes_data, dTolerance = 10000)`)`
#    could be improved to dynamically find the right amount of reduction that doesn't degrade resolution (country specific) 
#  - Text: check FR and finish the translations I missed!

## 0. Paths and Config

In [ ]:
# Set SNT Paths
SNT_ROOT_PATH  <- "~/workspace"
CODE_PATH      <- file.path(SNT_ROOT_PATH, "code")
CONFIG_PATH    <- file.path(SNT_ROOT_PATH, "configuration")
PIPELINE_PATH  <- file.path(SNT_ROOT_PATH, "pipelines", "snt_dhis2_formatting")

REPORTING_NB_PATH <- file.path(SNT_ROOT_PATH, "pipelines/snt_dhis2_formatting/reporting")

# Create output directories if they don't exist (before loading utils)
# figures_dir <- file.path(REPORTING_NB_PATH, "outputs", "figures")
FIGURES_OUT_PATH <- file.path(REPORTING_NB_PATH, "outputs", "figures")

In [ ]:
# Load util functions
source(file.path(CODE_PATH, "snt_utils.r"))
source(file.path(PIPELINE_PATH, "utils", "snt_dhis2_formatting_report.r"))

In [ ]:
# Create output directories if they don't exist (needs utils)
safe_create_dir(FIGURES_OUT_PATH) |> suppressMessages()

In [ ]:
required_packages <- c(
    "tidyverse", 
    "arrow", 
    "sf", 
    "reticulate",
    "scattermore" # for rasterizing ggplots
) 

# Execute function
install_and_load(required_packages)

In [ ]:
# # Set environment to load openhexa.sdk from the right environment

# ⚠️🧹 GP: Move this function to ./code/snt_utils.r !
# (do as separate branch/task, see https://bluesquare.atlassian.net/browse/SNT25-591 )
init_openhexa_env <- function(
  # In nb, run as `openhexa <- init_openhexa_env()`
  python_path = "/opt/conda/bin/python",
  proj_lib = "/opt/conda/share/proj",
  gdal_data = "/opt/conda/share/gdal"
) {
  Sys.setenv(PROJ_LIB = proj_lib)
  Sys.setenv(GDAL_DATA = gdal_data)
  Sys.setenv(RETICULATE_PYTHON = python_path)
  reticulate::py_config()$python

  return(reticulate::import("openhexa.sdk"))
}

openhexa <- init_openhexa_env()

In [ ]:
config_json <- load_snt_config(config_path = CONFIG_PATH)

In [ ]:
# Configuration variables
DATASET_NAME <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
COUNTRY_NAME <- config_json$SNT_CONFIG$COUNTRY_NAME
ADM_2 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_2)

# Get indicator variables to subset programmatically the dataset (instead of hardcoding col names)
INDICATOR_VARS <- config_json$DHIS2_DATA_DEFINITIONS$DHIS2_INDICATOR_DEFINITIONS |> names()

## Import data

### Routine data (raw from DHIS2 Extract)

In [ ]:
routine_data <- load_country_file_from_dataset(
    dataset_id = DATASET_NAME, 
    country_code = COUNTRY_CODE, 
    suffix = "_routine.parquet" 
    )
printdim(routine_data)

### Population data

In [ ]:
population_data <- load_country_file_from_dataset(
    dataset_id = DATASET_NAME, 
    country_code = COUNTRY_CODE, 
    suffix = "_population.parquet" 
    )

printdim(population_data)

### Shapes

In [ ]:
shapes_data <- load_country_file_from_dataset(
    dataset_id = DATASET_NAME, 
    country_code = COUNTRY_CODE, 
    suffix = "_shapes.geojson" 
    )

printdim(shapes_data) 

In [ ]:
# Simplify shapes to make it easier to render (fewer points per polygon)
# Set dTolerance bigger to keep fewer points
shapes_data_stsimplified <- st_simplify(shapes_data, dTolerance = 1000) # 10000
# GP:make this more refined to somehow reduce points without losing too much shape 
# (maybe use `rmapshaper::ms_simplify()` instead of `st_simplify()`) ... ?

cat("Simplifying shapes for faster rendering ...")
cat(paste0("\nOriginal shapes data points: ", sum(mapply(function(g) length(unlist(g)), sf::st_geometry(shapes_data))))) 
cat(paste0("\nSimplified shapes data points: ", sum(mapply(function(g) length(unlist(g)), sf::st_geometry(shapes_data_stsimplified)))))

# 1. Complétude du rapportage des indicateurs composites

## 1.1 Proportion de **formations sanitaires** ayant rapporté des valeurs nulles, manquantes (NULL) ou positives pour chaque indicateur

In [ ]:
# Step 0: Rename your data for convenience
data <- routine_data
# GP: find better name ... !

# Step 1: Convert PERIOD to DATE
data <- data %>%
  mutate(
    DATE = ymd(paste0(PERIOD, "01"))
  )

In [ ]:
long_data <- data %>%
  pivot_longer(cols = any_of(INDICATOR_VARS),
               names_to = "INDICATOR",
               values_to = "VALUE")

In [ ]:
# Step 3: Build expected full grid
full_grid <- expand_grid(
  OU_ID = unique(long_data$OU_ID),
  INDICATOR = unique(long_data$INDICATOR),
  DATE = unique(long_data$DATE)
)

In [ ]:
# Step 4: Join and assess reporting status
reporting_check <- full_grid %>%
  left_join(
    long_data %>% select(OU_ID, INDICATOR, DATE, VALUE),
    by = c("OU_ID", "INDICATOR", "DATE")
  ) %>%
  distinct() |> # GP added
  mutate(
    is_missing = is.na(VALUE),
    is_zero = VALUE == 0 & !is.na(VALUE),
    is_positive = VALUE > 0 & !is.na(VALUE)
  )

In [ ]:
# Step 5: Summarise reporting status
reporting_summary <- reporting_check %>%
  group_by(INDICATOR, DATE) %>% 
  summarise(
    n_total = n_distinct(OU_ID),
    n_missing = sum(is_missing),
    n_zero = sum(is_zero),
    n_positive = sum(is_positive),
    pct_missing = ifelse(n_total > 0, 100 * n_missing / n_total, 0),
    pct_zero = ifelse(n_total > 0, 100 * n_zero / n_total, 0),
    pct_positive = ifelse(n_total > 0, 100 * n_positive / n_total, 0),
    .groups = "drop"
  )

In [ ]:
# Check to see if n_total is = to the some of n_missing, n_zero and n_positive for each INDICATOR-DATE combo
if (nrow(reporting_summary %>% mutate(n_sum = (n_missing + n_zero + n_positive)) |>
    filter(n_sum > n_total)) > 0) {
  log_msg("Attention : Il existe des combinaisons INDICATOR-DATE où la somme de n_missing, n_zero et n_positive dépasse n_total. Veuillez vérifier le reporting_summary pour plus de détails.")
} 

In [ ]:
# Step 6: Prepare plot-ready data
plot_data <- reporting_summary %>%
  pivot_longer(
    cols = starts_with("pct_"),
    names_to = "Status",
    values_to = "Percentage"
  ) %>%
  mutate(
    Status = recode(Status,
                    pct_missing = "Valeur manquante",
                    pct_zero = "Valeur 0 rapportée", 
                    pct_positive = "Valeur ≥ 1 rapportée") # "positive" is confusing
  ) %>%
  complete(INDICATOR, DATE, Status, fill = list(Percentage = 0))

In [ ]:
plot <- ggplot(plot_data, aes(
  x = DATE, 
  y = Percentage, 
  # Reorder Status so that "Valeur ≥ 1 rapportée" is plotted at the bottom of y axis
  fill = forcats::fct_relevel(Status,  "Valeur manquante", "Valeur 0 rapportée", "Valeur ≥ 1 rapportée")
  )  ) +
  geom_col(position = "stack") +
  facet_wrap(~ INDICATOR, scales = "free_y", ncol = 4) +
  scale_y_continuous() +
  scale_fill_manual(values = c(
    "Valeur manquante" = "grey",
    "Valeur 0 rapportée" = "skyblue",
    "Valeur ≥ 1 rapportée" = "green"
  )) +
  labs(
    title = "Taux de rapportage par indicateur (niveau formation sanitaire)",
    subtitle = "Proportion des valeurs rapportées par mois et par indicateur",
    x = "Mois", y = "% des formations sanitaires",
    fill = "Statut du rapportage"
  ) +
  theme_minimal(base_size = 16) +
  theme(
    plot.title = element_text(face = "bold", size = 20),
    strip.text = element_text(size = 16),
    axis.title = element_text(size = 16),
    axis.text = element_text(size = 16)
  )

# Export plot with ggsave to figures_dir
plot_dir = file.path(FIGURES_OUT_PATH, "indicators_hf.png")
ggsave(
  filename = plot_dir,
  plot = plot,
  width = 50, height = 50, 
  units = "cm", 
  dpi = 300
) 

IRdisplay::display_png(file = plot_dir)

## 1.2 Proportion des districts ayant rapporté des valeurs manquantes (null), zéro (`0`) ou positives (`≥ 1`) pour chaque indicateur.

In [ ]:
# Step 3: Reshape to long format
data_long <- data %>%
  select(ADM2_ID, OU_ID, DATE, any_of(INDICATOR_VARS)) %>% 
  pivot_longer(cols = any_of(INDICATOR_VARS),
               names_to = "Indicator", values_to = "value") %>%
  mutate(value = as.numeric(value))

In [ ]:
# Step 4: Full expected grid at ADM2 level
full_grid <- expand_grid(
  ADM2_ID = unique(data_long$ADM2_ID),
  Indicator = unique(data_long$Indicator),
  DATE = unique(data_long$DATE)  # GP:was "Date"
)

In [ ]:
# Step 5: Detect if *any* health facility reported per district × indicator × date
reporting_check <- data_long %>%
  group_by(ADM2_ID, Indicator, DATE) %>% 
  summarise(
    is_missing = all(is.na(value)),
    is_zero = all(value == 0, na.rm = TRUE),
    is_positive = any(value > 0, na.rm = TRUE),
    .groups = "drop"
  )

In [ ]:
# Step 6: Join with full grid to fill in missing ADM2s
reporting_full <- full_grid %>%
  left_join(reporting_check, by = c("ADM2_ID", "Indicator", "DATE")) %>% # GP:was "Date"
  mutate(
    is_missing = replace_na(is_missing, TRUE),
    is_zero = replace_na(is_zero, FALSE),
    is_positive = replace_na(is_positive, FALSE)
  )

In [ ]:
# Step 7: Summarise by Indicator and Date
reporting_summary <- reporting_full %>%
  group_by(Indicator, DATE) %>%
  summarise(
    n_total = n_distinct(ADM2_ID),
    n_missing = sum(is_missing),
    n_zero = sum(is_zero & !is_missing),
    n_positive = sum(is_positive),
    pct_missing = ifelse(n_total > 0, 100 * n_missing / n_total, 0),
    pct_zero = ifelse(n_total > 0, 100 * n_zero / n_total, 0),
    pct_positive = ifelse(n_total > 0, 100 * n_positive / n_total, 0),
    .groups = "drop"
  )

In [ ]:
# Step 8: Reshape for plotting
plot_data <- reporting_summary %>%
  pivot_longer(cols = starts_with("pct_"),
               names_to = "Status", values_to = "Percentage") %>%
  mutate(Status = recode(Status,
                         pct_missing = "Valeur manquante",
                         pct_zero = "Valeur 0 rapportée", # GP: changed "zéro" to "nulle" 
                         pct_positive = "Valeur ≥ 1 rapportée")) %>%
  complete(Indicator, DATE, Status, fill = list(Percentage = 0))
# GP: werid that we still have the n_* cols (and they are very confusing ... !)

In [ ]:
# Step 9: Plot
plot <- ggplot(plot_data, aes(
  x = DATE, 
  y = Percentage, 
  fill = forcats::fct_relevel(Status,  "Valeur manquante", "Valeur 0 rapportée", "Valeur ≥ 1 rapportée")
  )) +
  geom_col(position = "stack") +
  facet_wrap(~ Indicator, scales = "free_y") +
  scale_y_continuous(limits = c(0, 100)) +
  scale_fill_manual(values = c(
    "Valeur manquante" = "grey",
    "Valeur 0 rapportée" = "skyblue",
    "Valeur ≥ 1 rapportée" = "green"
  )) +
  labs(
    title = "Taux de rapportage par indicateur (ADM2)",
    subtitle = "Proportion des districts (ADM2) rapportant chaque mois",
    x = "Mois", y = "% des ADM2",
    fill = "Statut du rapportage"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    plot.title = element_text(face = "bold", size = 18),
    strip.text = element_text(size = 14),
    axis.title = element_text(size = 14),
    axis.text = element_text(size = 12)
  )

plot_path = file.path(FIGURES_OUT_PATH, "indicators_adm2.png")
ggsave(
  filename = plot_path,
  plot = plot,
  width = 50, height = 50, 
  units = "cm", 
  dpi = 300
)

IRdisplay::display_png(file = plot_path)

# 2. Indicateurs composites

## 2.1. Cohérence interne des indicateurs composites

In [ ]:
# Step 1: Extract year and month from PERIOD
routine_hd_month <- routine_data %>% # GP
  mutate(
    YEAR = substr(PERIOD, 1, 4),
    MONTH = substr(PERIOD, 5, 6)
  ) %>%
  group_by(ADM2_ID, YEAR, MONTH) %>%
  summarise(across(all_of(INDICATOR_VARS), ~ sum(.x, na.rm = TRUE)),
    .groups = "drop"
  )

In [ ]:
# Now, we want to make 3 coherence plots: 
# 1. SUSP vs TEST (always data for this, as these are the "essential" indicators)
# 2. SUSP vs CONF (always data for this, as these are the "essential" indicators)
# 3. SUSP vs MALTREAT (might not have data for this, as MALTREAT is not an "essential" indicator)

In [ ]:
make_save_render_coherence_plot <- function(data,
                                             x_var,
                                             y_var,
                                             facet_var = NULL,
                                             clip_percentile = 0.999,
                                             point_size = 3,
                                             alpha = 0.4,
                                             raster_dpi = 350,
                                             output_dir = FIGURES_OUT_PATH,
                                             filename = NULL,
                                             width = 21,
                                             height = 21) {

  # Step 1: Summarise "red" points (y_var > x_var)
  n_total <- nrow(data)
  is_red <- data[[y_var]] > data[[x_var]]
  n_red <- sum(is_red, na.rm = TRUE)
  pct_red <- round(100 * n_red / n_total, 2)

  # Step 2: Determine a single shared axis limit (from combined x_var/y_var) so the view
  # is square and the dashed y=x line is at a 45° angle.
  clip_upper <- quantile(
    c(data[[x_var]], data[[y_var]]), 
    probs = clip_percentile, 
    na.rm = TRUE
    )
  n_clipped <- sum(data[[x_var]] > clip_upper | data[[y_var]] > clip_upper, na.rm = TRUE)
  pct_clipped <- round(100 * n_clipped / n_total, 3)

  title <- glue::glue("Coherence plot: {y_var} vs {x_var}")
  subtitle <- glue::glue(
    "Nombre de points de données : {format(n_total, big.mark = ' ')}\n",
    "Parmi ceux-ci, {format(n_red, big.mark = ' ')} points ({pct_red}%) se trouvent au-dessus de la ligne y = x (en rouge)\n",
    "Vue limitée au {clip_percentile * 100}e percentile (≤ {format(round(clip_upper), big.mark = ' ')}); ",
    "{format(n_clipped, big.mark = ' ')} points ({pct_clipped}%) se trouvent en dehors de cette plage et ne sont pas affichés."
  )

p <- ggplot(data, aes(x = .data[[x_var]], y = .data[[y_var]],
                        color = .data[[y_var]] > .data[[x_var]]))
p <- p + 
scattermore::geom_scattermore(
  alpha = alpha, 
  pointsize = point_size,
  pixels = c(raster_dpi * 2, raster_dpi * 2)
  )

  p <- p +
    scale_color_manual(
      values = c("TRUE" = "red", "FALSE" = "#1e81b0"), 
      name = glue::glue("{y_var} > {x_var}:")) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "black") +
    geom_hline(yintercept = 0) +
    geom_vline(xintercept = 0) +
    labs(title = title, subtitle = subtitle, x = x_var, y = y_var) +
    scale_x_continuous(expand = c(0, 0), labels = scales::comma) +
    scale_y_continuous(expand = c(0, 0), labels = scales::comma) +
    coord_equal(xlim = c(0, clip_upper), ylim = c(0, clip_upper)) +
    theme_minimal() +
    ggplot2::theme(
      plot.title = ggplot2::element_text(face = "bold"), # size = 10,
      plot.subtitle = ggplot2::element_text(size = 7),
      axis.title = ggplot2::element_text(size = 9, face = "bold"),
      axis.text = element_text(size = 7),
      legend.title = element_text(size = 10, face = "bold"), #  
      legend.text = ggplot2::element_text(size = 7),
      legend.position = "top",
      panel.grid.minor = element_blank()
    )

  # Step 4: Optional facetting (e.g. facet_var = "YEAR")
  # GP: not in use as the summary (in `subtitle`) does not account for faceting
  #     (so summaries are for full dataset, not grouped by any facet like YEAR)
  if (!is.null(facet_var)) {
    p <- p + facet_wrap(
      vars(.data[[facet_var]])
      )
  }

  # Step 5: Save to disk
  if (is.null(filename)) {
    filename <- glue::glue("{y_var}_vs_{x_var}.png")
  }
  plot_path <- file.path(output_dir, filename)
  ggsave(filename = plot_path, plot = p, width = width, height = height, units = "cm", dpi = raster_dpi)
  print(glue::glue("Le graphique de cohérence '{y_var} vs {x_var}' a été exporté vers : {plot_path}"), "info")

  IRdisplay::display_png(file = plot_path)
}


In [ ]:
make_save_render_coherence_plot(
    data = routine_hd_month,
    x_var = "SUSP",
    y_var = "TEST"
)

make_save_render_coherence_plot(
    data = routine_hd_month, 
    x_var = "TEST", 
    y_var = "CONF" 
    )

# Run only if routine_hd_month contains the col MALTREAT which is not a "mandatory" indicator
if ("MALTREAT" %in% colnames(routine_hd_month)) {
    make_save_render_coherence_plot(
        data = routine_hd_month, 
        x_var = "CONF", 
        y_var = "MALTREAT" 
    )
}

## 2.2. Tendence des indicateurs composites

In [ ]:
# Step 1: Aggregate monthly values
rds_clean_month <- routine_data %>%
  mutate(
    YEAR = substr(PERIOD, 1, 4),
    MONTH = substr(PERIOD, 5, 6),
    DATE = as.Date(paste(YEAR, MONTH, "01", sep = "-"))
  ) %>%
  group_by(YEAR, MONTH, DATE) %>%
  summarise(
    SUSP = sum(SUSP, na.rm = TRUE),
    TEST = sum(TEST, na.rm = TRUE),
    CONF = sum(CONF, na.rm = TRUE),
    PRES = sum(PRES, na.rm = TRUE),
    .groups = "drop"
  )

# Step 2: Plot monthly national trends
plot <- rds_clean_month %>%
  pivot_longer(cols = c(SUSP, TEST, CONF, PRES), names_to = "Indicator") %>%
  ggplot(aes(
    x = DATE, 
    y = value, 
    color = Indicator
    )) +
  geom_line(linewidth = 0.75) +
  labs(
    title = "Tendances mensuelles nationales des indicateurs composites",
    x = "Mois", y = "Nombre de cas", color = "Indicateur"
  ) +
  geom_hline(yintercept = 0, color = "grey50", linewidth = 0.5) +
  geom_vline(xintercept = min(rds_clean_month$DATE), color = "grey50", linewidth = 0.5) +
  scale_x_date(expand = c(0, 0), date_breaks = "1 year", date_labels = "%Y") +
  scale_y_continuous(expand = c(0, 0), labels = scales::comma) +
  theme_minimal(base_size = 9) +
  theme(
    plot.title = element_text(face = "bold", size = 10),
    axis.text.x = element_text(hjust = 0, face = "bold"),
    axis.title.x = element_blank(),
    panel.grid.minor.y = element_blank(),
    legend.position = "top"
  )

plot_dir <- file.path(FIGURES_OUT_PATH, "indicators_national_trends.png")
ggsave(
  filename = plot_dir, 
  plot = plot, 
  width = 21, 
  height = 21 / 2, 
  units = "cm", 
  dpi = 300
  )
print(glue::glue("Les tendances nationales des indicateurs composites ont été exportées vers : {plot_dir}"), "info")

IRdisplay::display_png(file = plot_dir)

## 2.3. Valeurs brutes des indicateurs composites desagregee

In [ ]:
# NOTE: Section 2.3. might need revision: see https://bluesquare.atlassian.net/browse/SNT25-638

In [ ]:
# 1. Identify disaggregated indicators (contain an underscore)
disaggr_vars <- INDICATOR_VARS[grepl("_", INDICATOR_VARS)]

# ----- Only execute if there are disaggregated indicators -----
if (length(disaggr_vars) != 0) {
  
# 2. Extract base indicator names (removes the last underscore and suffix)
base_vars <- unique(sub("_[^_]+$", "", disaggr_vars))

# 3. Combine disaggregated and matching base indicator names
target_indicators <- intersect(INDICATOR_VARS, c(disaggr_vars, base_vars))

# 4. Select and pivot
indicators_disaggr_data <- routine_data %>%
  select(
    YEAR, MONTH, 
    ends_with("_ID"), ends_with("_NAME"), 
    any_of(target_indicators)
  ) %>%
  pivot_longer(
    cols = any_of(target_indicators),
    names_to = "indicator", 
    values_to = "value"
  ) %>%
  mutate(value = as.numeric(value)) |>
  dplyr::mutate(group = sub("_.*", "", indicator)) # JUST ADDED for BDI 

}

# head(indicators_disaggr_data, 3)

In [ ]:
# Only execute if there are disaggregated indicators
if (length(disaggr_vars) != 0) {

nr_unique_year <- length(unique(indicators_disaggr_data$YEAR))
calc_width <- max(10, nr_unique_year * 5)

nr_unique_adm2 <- length(unique(indicators_disaggr_data$ADM2_ID))
nr_unique_groups <- length(unique(indicators_disaggr_data$group))
# (max ggsave size: 50 inches = 127 cm)
calc_height <- min(126, (ceiling(nr_unique_adm2 / 3) * nr_unique_groups))


plot_ind_disaggr <- indicators_disaggr_data %>%
group_by(YEAR, ADM2_ID, ADM2_NAME, ADM1_NAME, group, indicator) |>
summarise(value = sum(value, na.rm = TRUE), .groups = "drop") %>%
  ggplot(aes(x = value, y = fct_rev(ADM2_NAME), fill = indicator)) +
  geom_bar(stat = "identity") +
  scale_x_continuous(labels = scales::label_number(big.mark = ",")) +
  scale_fill_viridis_d(option = "C") +
  facet_grid(
  rows = vars(ADM1_NAME, group),
  cols = vars(YEAR),
  scales = "free_y",
  space = "free_y",
  switch = "y"
) +
  theme_minimal() +
  theme(
    plot.subtitle = element_text(margin = margin(0, 0, 20, 0)),
    legend.position = "bottom",
    legend.title = element_blank(),
    legend.key.height = unit(0.25, "cm"),
    axis.text.x = element_text(size = 7, angle = 90),
    axis.text.y = element_text(size = 7),
    axis.title = element_blank(),
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_blank(),
    strip.placement = "outside",
    strip.text = element_text(face = "bold", size = 10)
  ) +
  guides(fill = guide_legend(nrow = 1)) 

ind_disaggr_path <- file.path(FIGURES_OUT_PATH, "ind_disaggr_counts.png")
ggsave(
  filename = ind_disaggr_path,
  plot = plot_ind_disaggr,
  bg = "white",
  width = calc_width, 
  height = calc_height, 
  units = "cm",
  dpi = 200
)

print(glue::glue("Le graphique des indicateurs de population désagrégués a été exporté vers : {ind_disaggr_path}"), "info")

IRdisplay::display_png(file = ind_disaggr_path)

}

In [ ]:
# Same plot as above, but as proportions (0-100%) instead of raw counts
if (length(disaggr_vars) != 0) {

plot_ind_disaggr_prop <- indicators_disaggr_data %>%
group_by(YEAR, ADM2_ID, ADM2_NAME, ADM1_NAME, group, indicator) |>
summarise(value = sum(value, na.rm = TRUE), .groups = "drop") %>%
  ggplot(aes(
    x = value, 
    y = fct_rev(ADM2_NAME), 
    fill = indicator)) +
  geom_bar(stat = "identity", position = "fill") +
  scale_x_continuous(
    labels = scales::percent_format(),
    breaks = c(0, 0.25, 0.5, 0.75, 1),
    expand = c(0, 0)
    ) +
  scale_fill_viridis_d(option = "C") +
  facet_grid(
    rows = vars(ADM1_NAME, group),
    cols = vars(YEAR),
    scales = "free_y",
    space = "free_y",
    switch = "y"
  ) +
  theme_minimal() +
  theme(
    plot.subtitle = element_text(margin = margin(0, 0, 20, 0)),
    legend.position = "bottom",
    legend.title = element_blank(),
    legend.key.height = unit(0.25, "cm"),
    axis.text.x = element_text(size = 7, angle = 90),
    axis.text.y = element_text(size = 7),
    axis.title = element_blank(),
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_blank(),
    strip.placement = "outside",
    strip.text = element_text(face = "bold", size = 10)
  ) +
  guides(fill = guide_legend(nrow = 1))

ind_disaggr_prop_path <- file.path(FIGURES_OUT_PATH, "ind_disaggr_proportions.png")
ggsave(
  filename = ind_disaggr_prop_path,
  plot = plot_ind_disaggr_prop,
  bg = "white",
  width = calc_width,
  height = calc_height,
  units = "cm",
  dpi = 200
)

print(glue::glue("Le graphique des proportions des indicateurs désagrégués a été exporté vers : {ind_disaggr_prop_path}"), "info")

IRdisplay::display_png(file = ind_disaggr_prop_path)

}

# 3. **Populations** pour ADM2 

In [ ]:
# Set fig_width and fig_heght for all pop plots (maps, hist, scatterplots ... )
fig_width = 21
fig_height = fig_width

## 3.1. Vue d'ensemble: Cartes

In [ ]:
# Check which nodes have data (de id) in snt_config$DHIS2_DATA_DEFINITIONS$POPULATION_INDICATOR_DEFINITIONS

# Define list of accepted population indicators (as per config file)
pop_indicators_defs <- config_json$DHIS2_DATA_DEFINITIONS$POPULATION_INDICATOR_DEFINITIONS
pop_indicators <- pop_indicators_defs |> names()

# Check which pop_indicators have the subnode $ids non empty (filled with an alphanumeric string)
pop_indicators_with_ids <- pop_indicators[
  sapply(pop_indicators, function(ind) {
    ids <- pop_indicators_defs[[ind]]$ids
    length(ids) > 0 && any(grepl("^[[:alnum:]]+$", ids))
  })
]

In [ ]:
map_data <- shapes_data_stsimplified %>%
    left_join(population_data, by = "ADM2_ID")

In [ ]:
plot_and_save_map <- function(pop_col, 
                              data = map_data, 
                              output_dir = FIGURES_OUT_PATH, 
                              width = fig_width, 
                              height = fig_height) {
  
  # Generate plot
  plot <- ggplot(data) +
    geom_sf(aes(fill = .data[[pop_col]]), color = "white", linewidth = 0.2) +
    scale_fill_viridis_c(option = "C", name = pop_col) +
    labs(title = pop_col) +
    theme_minimal(base_size = 14)
  
  # Define file path and save
  figure_path <- file.path(output_dir, glue::glue("{pop_col}_map.png"))
  
  ggsave(
    filename = figure_path, 
    plot = plot, 
    width = width, 
    height = height, 
    units = "cm", 
    dpi = 300
  )
  # Add log message in French
  print(glue::glue("Carte de population pour {pop_col} exportée sous: {figure_path}"), "info")

  # Display in notebook
  IRdisplay::display_png(file = figure_path)
}

# Iterate over every indicator column name
purrr::walk(pop_indicators_with_ids, plot_and_save_map)

## 3.2. <b>Qualité</b> des données

### 3.2.1. Distribution des valeurs de population

In [ ]:
# outlier_cutoff = 0.995

# plot_and_save_hist <- function(pop_col, 
#                                data = population_data, 
#                                output_dir = FIGURES_OUT_PATH, 
#                                width = fig_width, 
#                                height = fig_height) {
  
#   # Calculate per-year outlier flags
#   plot_data <- data %>%
#     group_by(YEAR) %>%
#     mutate(
#       is_outlier = .data[[pop_col]] > quantile(.data[[pop_col]], outlier_cutoff, na.rm = TRUE)
#     ) %>%
#     ungroup()

#   # Compute per-year cutoff values string
# cutoffs_str <- plot_data %>%
#   group_by(YEAR) %>%
#   summarize(cutoff = quantile(.data[[pop_col]], 0.995, na.rm = TRUE), .groups = "drop") %>%
#   mutate(txt = glue::glue("{YEAR} : {format(round(cutoff, 1), big.mark = ' ')}")) %>%
#   pull(txt) %>%
#   paste(collapse = " | ")

#   # Generate histogram with outside rug
#   hist <- ggplot(data = plot_data) +
#     geom_rug(
#       aes(
#         x = .data[[pop_col]], 
#         color = .data[["is_outlier"]]
#       ),
#       sides = "b",
#       outside = FALSE, #TRUE
#       length = unit(1, "npc"),
#       linewidth = 0.5,
#       alpha = 0.75
#     ) +
#     geom_histogram(
#       aes(x = .data[[pop_col]]), 
#       bins = 50, 
#       fill = "grey21", 
#       color = "black", 
#       alpha = 0.75,
#       linewidth = 0.1
#     ) +
#     geom_hline(yintercept = 0, color = "grey21") +
#     scale_color_manual(
#       values = c("FALSE" = "skyblue", "TRUE" = "firebrick"),
#       guide = "none" # Prevents generating an extra legend for rug colors
#     ) +
#     # scale_x_continuous(expand = c(0, 0)) + 
#     scale_x_continuous(expand = c(0, 0), labels = scales::comma) + 
#     scale_y_continuous(expand = c(0, 0)) + 
#     # scale_x_continuous(labels = scales::comma) +
#     facet_grid(YEAR ~ ., scales = "free_y") +
#     coord_cartesian(clip = "off") + # Allows geom_rug to render outside the plot panel
#     labs(
#       title = glue::glue("Distribution des valeurs de {pop_col} pour ADM2 pour ANNEE"), 
#       subtitle = glue::glue(
#     "Les lignes verticales rouges indiquent des valeurs suspectes (supérieures au {outlier_cutoff}e centile par année).\n" ),
#       x = pop_col,
#       y = "Comptage (ADM2)",
#     ) +
#     theme_minimal(base_size = 9) +
#     theme(
#       axis.title.x = element_text(margin = margin(t = 12)), 
#       # axis.title.y = element_blank(),
#       axis.text.y = element_blank(),
#       plot.margin = margin(t = 10, r = 10, b = 15, l = 10),
#       panel.grid.minor = element_blank(),
#       panel.grid.major.y = element_blank()
#     )

#   hist_path <- file.path(output_dir, glue::glue("{pop_col}_hist.png"))
  
#   ggsave(
#     filename = hist_path,
#     plot = hist,
#     width = width,
#     height = width / 3 * length(unique(data$YEAR)),
#     units = "cm",
#     dpi = 300
#   )
  
#   log_msg(glue::glue("Histogramme de distribution de la population pour {pop_col} exporté sous: {hist_path}"), "info")
#   IRdisplay::display_png(file = hist_path)
# }

# # Iterate over all column names
# purrr::walk(pop_indicators_with_ids, plot_and_save_hist)

In [ ]:
# Same histogram as above, but without geom_rug()
plot_and_save_hist <- function(pop_col, 
                                      data = population_data, 
                                      output_dir = FIGURES_OUT_PATH, 
                                      width = fig_width, 
                                      height = fig_height) {

  # Generate histogram
  hist <- ggplot(data = data) +
    geom_histogram(
      aes(x = .data[[pop_col]]), 
      bins = 50, 
      fill = "grey21", 
      color = "black", 
      alpha = 0.75,
      linewidth = 0.1
    ) +
    geom_hline(yintercept = 0, color = "grey21") +
    scale_x_continuous(expand = c(0, 0), labels = scales::comma) + 
    scale_y_continuous(expand = c(0, 0)) + 
    facet_wrap(~YEAR,nrow = 1) +
    labs(
      title = glue::glue("Distribution des valeurs de {pop_col} pour ADM2 pour ANNEE"), 
      x = pop_col,
      y = "Comptage (ADM2)",
    ) +
    theme_minimal(base_size = 9) +
    theme(
      axis.title.x = element_text(margin = margin(t = 12)), 
      axis.text.y = element_blank(),
      plot.margin = margin(t = 10, r = 10, b = 15, l = 10),
      panel.grid.minor = element_blank(),
      panel.grid.major.y = element_blank(),
      strip.background = element_rect(colour = "grey21")
    )

  hist_path <- file.path(output_dir, glue::glue("{pop_col}_hist.png"))
  
  ggsave(
    filename = hist_path,
    plot = hist,
    width = width / 3 * length(unique(data$YEAR)),
    height = width / 3,
    units = "cm",
    dpi = 300
  )
  
  print(glue::glue("Histogramme de distribution de la population pour {pop_col} exporté sous: {hist_path}"), "info")
  IRdisplay::display_png(file = hist_path)
}

# Iterate over all column names
purrr::walk(pop_indicators_with_ids, plot_and_save_hist)

### 3.2.2. Valeurs brutes de population

In [ ]:
# Create vector of pop cols present in population_data
# (to know which cols to use in ggplot in next code cell)
POPULATION_INDICATORS <- config_json$DHIS2_DATA_DEFINITIONS$POPULATION_INDICATOR_DEFINITIONS |> names()
POPULATION_INDICATORS_IN_POP_DATA <- intersect(POPULATION_INDICATORS, names(population_data))

In [ ]:
nr_unique_year <- length(unique(population_data$YEAR))
calc_width <- max(10, nr_unique_year * 5)
nr_unique_adm2 <- length(unique(population_data$ADM2_ID))
# (max ggsave size: 50 inches = 127 cm)
calc_height <- min(126, ceiling(nr_unique_adm2 / 3))


plot_pop_disaggr <- population_data %>%
  pivot_longer(
    cols = all_of(POPULATION_INDICATORS_IN_POP_DATA),
    names_to = "indicator",
    values_to = "value"
    ) |>
  mutate(indicator = factor(indicator, levels = POPULATION_INDICATORS_IN_POP_DATA)) |>
  ggplot(aes(x = value, y = fct_rev(ADM2_NAME), fill = fct_rev(indicator))) +
  geom_bar(stat = "identity") +
  scale_x_continuous(labels = scales::label_number(big.mark = ",")) +
  scale_fill_viridis_d(option = "C") +
  facet_grid(
    cols = vars(YEAR), rows = vars(ADM1_NAME),
    scales = "free_y", # Keep X fixed so I can compare across years
    space = "free_y", switch = "y") +
  theme_minimal() +
  theme(
    plot.subtitle = element_text(margin = margin(0, 0, 20, 0)),
    legend.position = "bottom",
    legend.title = element_blank(),
    legend.key.height = unit(0.25, "cm"),
    axis.text.x = element_text(size = 7, angle = 90),
    axis.text.y = element_text(size = 7),
    axis.title = element_blank(),
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_blank(),
    strip.placement = "outside",
    strip.text = element_text(face = "bold", size = 10)
  ) +
  guides(fill = guide_legend(nrow = 1))

pop_disaggr_path <- file.path(FIGURES_OUT_PATH, "pop_disaggr_counts.png")
ggsave(
  filename = pop_disaggr_path,
  plot = plot_pop_disaggr,
  bg = "white",
  width = calc_width, 
  height = calc_height, 
  units = "cm",
  dpi = 200
)

print(glue::glue("Le graphique des indicateurs de population désagrégués a été exporté vers : {pop_disaggr_path}"), "info")

IRdisplay::display_png(file = pop_disaggr_path)

In [ ]:
# Only render this plot of there is disaggregated population data
# (if only col `POPULATION` is present, then the plot will 
#  just be 100% length bars of one color ... useless)
if (length(POPULATION_INDICATORS_IN_POP_DATA) > 1) {

plot_pop_disaggr_prop <- population_data %>%
  pivot_longer(
    cols = all_of(POPULATION_INDICATORS_IN_POP_DATA),
    names_to = "indicator",
    values_to = "value"
    ) |>
  mutate(indicator = factor(indicator, levels = POPULATION_INDICATORS_IN_POP_DATA)) |>
  ggplot(aes(x = value, y = fct_rev(ADM2_NAME), fill = fct_rev(indicator))) +
  geom_bar(stat = "identity", position = "fill") +
  scale_x_continuous(
    labels = scales::percent_format(),
    breaks = c(0, 0.25, 0.5, 0.75, 1),
    expand = c(0, 0)
    ) +
  scale_fill_viridis_d(option = "C") +
  facet_grid(
    cols = vars(YEAR), rows = vars(ADM1_NAME),
    scales = "free_y", # Keep X fixed so I can compare across years
    space = "free_y", switch = "y") +
  theme_minimal() +
  theme(
    plot.subtitle = element_text(margin = margin(0, 0, 20, 0)),
    legend.position = "bottom",
    legend.title = element_blank(),
    legend.key.height = unit(0.25, "cm"),
    axis.text.x = element_text(size = 7, angle = 90),
    axis.text.y = element_text(size = 7),
    axis.title = element_blank(),
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_blank(),
    strip.placement = "outside",
    strip.text = element_text(face = "bold", size = 10)
  ) +
  guides(fill = guide_legend(nrow = 1))

pop_disaggr_prop_path <- file.path(FIGURES_OUT_PATH, "pop_disaggr_proportions.png")
ggsave(
  filename = pop_disaggr_prop_path,
  plot = plot_pop_disaggr_prop,
  bg = "white",
  width = calc_width,
  height = calc_height,
  units = "cm",
  dpi = 200
)

print(glue::glue("Le graphique des proportions des indicateurs de population désagrégués a été exporté vers : {pop_disaggr_prop_path}"), "info")

IRdisplay::display_png(file = pop_disaggr_prop_path)

}

### 3.2.3. Évolution des valuers de population par année

In [ ]:
plot_and_save_scatter <- function(pop_col, 
                                  data = population_data, 
                                  output_dir = FIGURES_OUT_PATH, 
                                  width = fig_width) {
  
  # Calculate height dynamically based on number of ADM2 units
  nr_unique_adm2 <- length(unique(data$ADM2_NAME))
  calc_height <- max(10, nr_unique_adm2 / 5)
  
  # Generate scatter plot
  scatter <- ggplot(data) +
    geom_point(
      aes(
        x = .data[[pop_col]],
        y = forcats::fct_reorder(ADM2_NAME, .data[[pop_col]]),
        color = factor(YEAR)
      )
    ) +
    facet_grid(
      rows = vars(ADM1_NAME), 
      scales = "free_y", 
      space = "free_y", 
      switch = "y"
    ) +
    scale_x_continuous(labels = scales::comma) +
    scale_color_viridis_d(option = "mako", end = 0.8, guide = guide_legend(nrow = 1)) +
    labs(
      title = glue::glue("Évolution de {pop_col} par année"),
      subtitle = "Répartition démographique par ADM1 et ADM2. Les points plus clairs indiquent les années les plus récentes",
      x = pop_col,
      color = "Année:"
    ) +
    theme_minimal() +
    theme(
      axis.text = element_text(size = 5),
      axis.title.x = element_text(size = 7),
      axis.title.y = element_blank(),
      strip.text.y = element_text(size = 7),
      strip.placement = "outside",
      panel.grid.minor.x = element_blank(),
      legend.position = "top",
      legend.title = element_text(size = 7),
      legend.text = element_text(size = 7),
      plot.title = element_text(face = "bold", size = 9, margin = margin(b = 3)),
      plot.subtitle = element_text(size = 7, color = "gray30", margin = margin(b = 6))
    )

  # Define file path and save
  plot_path <- file.path(output_dir, glue::glue("{pop_col}_scatter.png"))
  
  ggsave(
    filename = plot_path,
    plot = scatter,
    width = width,
    height = calc_height,
    units = "cm",
    dpi = 300
  )
  
  print(glue::glue("Graphique de l'évolution de {pop_col} exporté sous: {plot_path}"), "info")
  IRdisplay::display_png(file = plot_path)
}

# Iterate over all indicator columns
purrr::walk(pop_indicators_with_ids, plot_and_save_scatter)